# 2. Data Preprocessing

This notebook transforms the raw Facebook posts dataset into a validated numerical feature matrix suitable for dimensionality reduction and clustering.

## 2.1. Environment and Library Imports

In [1]:
# Standard library imports
import logging
import sys
from pathlib import Path

# Third-party imports
import joblib
import numpy as np
import pandas as pd

In [2]:
current_directory = Path.cwd().resolve()
project_candidates = (current_directory, *current_directory.parents)

PROJECT_ROOT = next(
    (path for path in project_candidates if (path / "src").is_dir()),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the project root directory.")

SRC_DIRECTORY = PROJECT_ROOT / "src"

if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

LOGGER = logging.getLogger("facebook_clustering.data_preprocessing")

### 2.1.1. Custom Utilities

In [ ]:
from data_collection_utils import (
    dataframe_overview,
    save_dataframe_csv,
    validate_dataframe_contract,
)
from data_preprocessing_utils import (
    add_cyclical_features,
    add_datetime_features,
    build_clustering_preprocessor,
    component_consistency_summary,
    validate_nonnegative_columns,
    validate_numeric_matrix,
)

## 2.2. Configuration

All paths and feature groups are defined in one place to make the workflow easier to maintain and audit.

In [ ]:
RAW_DATA_PATH = (
    PROJECT_ROOT / "data" / "raw" / "facebook_live_sellers.csv"
)
CLEANED_DATA_PATH = (
    PROJECT_ROOT / "data" / "processed" / "facebook_posts_cleaned.csv"
)
MODEL_MATRIX_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "facebook_posts_model_matrix.csv"
)
PREPROCESSOR_PATH = PROJECT_ROOT / "artifacts" / "preprocessor.joblib"

EXPECTED_ROW_COUNT = 7_050
DATETIME_COLUMN = "status_published"
CATEGORICAL_COLUMNS = ["status_type"]
REACTION_COMPONENT_COLUMNS = [
    "num_likes",
    "num_loves",
    "num_wows",
    "num_hahas",
    "num_sads",
    "num_angrys",
]
ENGAGEMENT_COLUMNS = [
    "num_comments",
    "num_shares",
    *REACTION_COMPONENT_COLUMNS,
]
REQUIRED_COLUMNS = {
    DATETIME_COLUMN,
    *CATEGORICAL_COLUMNS,
    "num_reactions",
    *ENGAGEMENT_COLUMNS,
}

## 2.3. Raw Data Loading

This notebook consumes the immutable CSV artifact generated by `01_data_collection_and_understanding.ipynb`. It does not download the dataset again.

In [ ]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        "Raw dataset not found. Run notebook "
        "01_data_collection_and_understanding.ipynb first."
    )

raw_posts = pd.read_csv(RAW_DATA_PATH)

validate_dataframe_contract(
    raw_posts,
    required_columns=REQUIRED_COLUMNS,
)

if len(raw_posts) != EXPECTED_ROW_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_ROW_COUNT:,} rows, "
        f"but found {len(raw_posts):,}."
    )

LOGGER.info("Dataset loaded from: %s", RAW_DATA_PATH)
LOGGER.info("Dataset shape: %s", raw_posts.shape)
display(dataframe_overview(raw_posts))

## 2.4. Data Quality Assessment

Missing values, negative engagement counts, and exact duplicate feature rows are quantified before any transformation is applied.

In [ ]:
numeric_columns = raw_posts.select_dtypes(include="number").columns
missing_value_count = int(raw_posts.isna().sum().sum())
negative_counts = validate_nonnegative_columns(
    raw_posts,
    numeric_columns,
)
duplicate_count = int(raw_posts.duplicated().sum())

if missing_value_count:
    raise ValueError(f"Found {missing_value_count} missing values.")

quality_summary = pd.Series(
    {
        "rows": len(raw_posts),
        "columns": raw_posts.shape[1],
        "missing_values": missing_value_count,
        "negative_numeric_values": int(negative_counts.sum()),
        "exact_duplicate_rows": duplicate_count,
        "duplicate_percentage": duplicate_count / len(raw_posts) * 100,
    },
    name="value",
)

display(quality_summary.to_frame())

The dataset contains 54 exact feature duplicates. They are preserved because the available feature table does not include the original post identifier or seller identifier. Identical feature values are therefore not sufficient evidence that the observations represent duplicated records. If identifiers become available later, this decision should be reassessed.

## 2.5. Temporal Feature Engineering

The original publication timestamp is parsed and expanded into interpretable calendar features. Month, weekday, and hour are also encoded as cyclical coordinates so adjacent boundary values remain close in the model space.

In [ ]:
cleaned_posts = add_datetime_features(
    raw_posts,
    DATETIME_COLUMN,
    datetime_format="%m/%d/%Y %H:%M",
    parsed_column="published_at",
    feature_prefix="publication",
)

LOGGER.info(
    "Publication range: %s to %s",
    cleaned_posts["published_at"].min(),
    cleaned_posts["published_at"].max(),
)

### 2.5.1. Cyclical Encoding

In [ ]:
model_features = add_cyclical_features(
    cleaned_posts,
    feature_periods={
        "publication_month": 12,
        "publication_weekday_number": 7,
        "publication_hour": 24,
    },
    offsets={"publication_month": 1},
)

In [ ]:
CYCLICAL_COLUMNS = [
    "publication_month_sin",
    "publication_month_cos",
    "publication_weekday_number_sin",
    "publication_weekday_number_cos",
    "publication_hour_sin",
    "publication_hour_cos",
]

display(model_features[CYCLICAL_COLUMNS].head())

## 2.6. Reaction Consistency and Redundancy

`num_reactions` is compared with the sum of the individual reaction columns before the modeling feature set is defined.

In [ ]:
reaction_consistency = component_consistency_summary(
    cleaned_posts,
    total_column="num_reactions",
    component_columns=REACTION_COMPONENT_COLUMNS,
)

display(reaction_consistency.to_frame())

`num_reactions` is retained in the cleaned dataset for interpretation but excluded from the clustering feature matrix. It is almost entirely derived from the individual reaction columns, so including both would give reaction activity duplicated weight in distance calculations.

## 2.7. Modeling Feature Matrix

A synthetic `record_id` is created only to align saved artifacts. It is never passed to PCA or a clustering algorithm.

In [ ]:
cleaned_posts.insert(0, "record_id", np.arange(len(cleaned_posts)))

TEMPORAL_MODEL_COLUMNS = [
    "publication_year",
    "publication_month_sin",
    "publication_month_cos",
    "publication_weekday_number_sin",
    "publication_weekday_number_cos",
    "publication_hour_sin",
    "publication_hour_cos",
]
MODEL_INPUT_COLUMNS = [
    *ENGAGEMENT_COLUMNS,
    *CATEGORICAL_COLUMNS,
    *TEMPORAL_MODEL_COLUMNS,
]

model_input = model_features[MODEL_INPUT_COLUMNS].copy()
LOGGER.info("Model input shape: %s", model_input.shape)

### 2.7.1. Preprocessing Pipelines

Engagement counts receive a `log1p` transformation before standardization because they are non-negative and strongly right-skewed. Temporal features are standardized separately. Post type is one-hot encoded without dropping a reference category, preserving a symmetric representation for distance-based methods.

In [ ]:
preprocessor = build_clustering_preprocessor(
    log_scaled_columns=ENGAGEMENT_COLUMNS,
    standard_scaled_columns=TEMPORAL_MODEL_COLUMNS,
    categorical_columns=CATEGORICAL_COLUMNS,
)

In [ ]:
model_matrix = preprocessor.fit_transform(model_input)
model_matrix.insert(
    0,
    "record_id",
    cleaned_posts["record_id"].to_numpy(),
)

display(model_matrix.head())

## 2.8. Post-Transformation Validation

In [ ]:
model_values = validate_numeric_matrix(
    model_matrix,
    excluded_columns=["record_id"],
    expected_row_count=len(cleaned_posts),
)

if not model_matrix["record_id"].equals(cleaned_posts["record_id"]):
    raise ValueError("The record identifiers are not aligned.")

LOGGER.info("Preprocessed model matrix validated successfully.")
LOGGER.info("Model matrix shape: %s", model_matrix.shape)

In [ ]:
transformation_summary = pd.DataFrame(
    {
        "feature": model_values.columns,
        "dtype": model_values.dtypes.astype(str).to_numpy(),
        "mean": model_values.mean().to_numpy(),
        "std": model_values.std().to_numpy(),
        "minimum": model_values.min().to_numpy(),
        "maximum": model_values.max().to_numpy(),
    }
)

display(transformation_summary)

## 2.9. Artifact Persistence

The cleaned dataset remains interpretable, while the model matrix contains only numerical features. The fitted preprocessor is stored so that the transformation can be reproduced without refitting.

In [ ]:
cleaned_posts = cleaned_posts.drop(columns=[DATETIME_COLUMN])

cleaned_data_path = save_dataframe_csv(
    cleaned_posts,
    CLEANED_DATA_PATH,
)
model_matrix_path = save_dataframe_csv(
    model_matrix,
    MODEL_MATRIX_PATH,
)
PREPROCESSOR_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(preprocessor, PREPROCESSOR_PATH)

LOGGER.info("Cleaned dataset saved to: %s", cleaned_data_path)
LOGGER.info("Model matrix saved to: %s", model_matrix_path)
LOGGER.info("Fitted preprocessor saved to: %s", PREPROCESSOR_PATH)